# VERIFACT-Lite on a Colab T4

The laptop run was shaped by a 4 GB GPU: two 3B judges could not stay resident, so the pipeline unloads one judge before loading the next. A free Colab T4 has 16 GB, which removes that constraint and lets the judges be 7–8B instead of 3B.

Two experiments, in order:

| | Primary model | Judges | What it tests |
|---|---|---|---|
| **A** | Qwen2.5-3B (unchanged) | Qwen2.5-7B + Llama-3.1-8B | Detection quality alone. Extraction replays from cache, so the **baseline fact set is byte-identical** to the laptop run and only the judges differ. |
| **B** | Qwen2.5-7B | Qwen2.5-7B + Llama-3.1-8B | The whole system unconstrained. The baseline itself changes, so this is a **new system, not a reproduction**. |

Both reuse `outputs/answers.json` — the answers under evaluation were generated once on the laptop and are never regenerated. Without that, nothing would be comparable.

**Runtime → Change runtime type → T4 GPU** before running anything.

Rough timings on a T4: A ≈ 60–90 min, B ≈ 90–150 min. Use `LIMIT` below to shorten either.

## 1 · Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
if not out:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")
vram = int(out.splitlines()[0])
print(f"\n{vram} MiB VRAM")
print("Both 7-8B judges can stay resident." if vram >= 14000 else
      "Under 14 GB: set run.unload_between_judges: true in the config before running.")

## 2 · Upload the bundle

Build it on the laptop first:

```bash
python colab/make_bundle.py
```

That writes `colab/verifact_lite_bundle.zip` (~45 MB): the code, the benchmark, the generated answers, the LLM call cache and the cached Wikipedia pages.

In [ ]:
import os, zipfile, pathlib
from google.colab import files

WORK = pathlib.Path("/content/verifact-lite")
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

if not (WORK / "src").exists():
    uploaded = files.upload()                     # pick verifact_lite_bundle.zip
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as zf:
        zf.extractall(WORK)
    os.remove(name)

print("cwd:", os.getcwd())
for p in ["src", "experiments", "data/challenge_set.jsonl", "data/mini_reference.jsonl",
          "outputs/answers.json", "cache", "data/evidence"]:
    path = WORK / p
    mark = "ok " if path.exists() else "MISSING"
    extra = f"({sum(1 for _ in path.rglob('*.json'))} files)" if path.is_dir() else ""
    print(f"  {mark} {p} {extra}")

## 3 · Install dependencies and Ollama

In [ ]:
!pip install -q pyyaml requests pytest

# zstd: the Ollama installer needs it to extract, and Colab's image has none.
# pciutils/lshw: without them the installer cannot detect the GPU and may skip
# the CUDA dependencies, leaving every model on the CPU.
!apt-get -qq install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

!which ollama && ollama --version

In [ ]:
import subprocess, time, requests

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(60):
    try:
        requests.get("http://127.0.0.1:11434/api/version", timeout=2).raise_for_status()
        print("ollama is up:", requests.get("http://127.0.0.1:11434/api/version").json())
        break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("ollama did not start")

## 4 · Pull the models

About 10 GB total, a few minutes on Colab's network. Qwen2.5-3B is needed for experiment A, where it stays the extractor and refiner.

In [ ]:
MODELS = [
    "qwen2.5:3b-instruct-q4_K_M",   # experiment A keeps this as the primary model
    "qwen2.5:7b-instruct-q4_K_M",   # judge 1, and the primary model in experiment B
    "llama3.1:8b-instruct-q4_K_M",  # judge 2
]
for m in MODELS:
    print(f"
=== pulling {m} ===")
    !ollama pull {m}

!ollama list

# A model served from the CPU is roughly 10x slower, which turns a 90 minute
# run into an overnight one. Load one and check where it actually landed.
!ollama run qwen2.5:3b-instruct-q4_K_M "hi" > /dev/null 2>&1
import subprocess
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
print(ps)
if "GPU" not in ps:
    raise SystemExit(
        "Model is not on the GPU. Run:
"
        "  !apt-get -qq install -y pciutils lshw && curl -fsSL https://ollama.com/install.sh | sh
"
        "then rerun the 'start the server' cell and this one."
    )

## 5 · Sanity check: tests pass and every model answers

In [ ]:
!python -m pytest tests -q

from src.utils import load_config
from src.model_client import build_primary, build_judges, probe

for cfg_path in ["colab/config.judges7b.yaml", "colab/config.full7b.yaml"]:
    cfg = load_config(cfg_path)
    print(f"\n{cfg_path}")
    print("  primary:", cfg["model"]["model_id"], "->", probe(build_primary(cfg)))
    for j in build_judges(cfg):
        print(f"  judge {j.alias}:", j.model_id, "->", probe(j))

## 6 · Experiment A — judge upgrade only

Extraction, refinement, verification and retrieval all still run on Qwen2.5-3B, and the cached calls replay for free. The judges are 7B and 8B. Results land in `outputs_colab_a/`.

Set `LIMIT` to a number to run only the first N questions.

In [ ]:
LIMIT = ""            # e.g. "--limit 4" for a quick look; "" runs all 12
CFG_A = "colab/config.judges7b.yaml"

import time
t0 = time.time()
!python -u -m experiments.run_ablation --config {CFG_A} {LIMIT}
print(f"\nablation (union) took {(time.time()-t0)/60:.1f} min")

In [ ]:
# Majority vote reuses the judge decisions already cached above, so this is quick.
!python -u -m experiments.run_ablation --config {CFG_A} --ensemble majority {LIMIT}
!python -u -m experiments.run_baseline --config {CFG_A} {LIMIT}
!python -u -m experiments.run_verifact --config {CFG_A} {LIMIT}
!python -u -m experiments.compare --config {CFG_A}

### A vs the laptop run

In [ ]:
!python colab/compare_runs.py laptop-3B=outputs/summary.json judges-7B=outputs_colab_a/summary.json

**What to look for in A.** The baseline column should be unchanged — same facts, same verdicts — because nothing on that path moved. If it did change, something other than the judges varied and the comparison is not clean.

In the VERIFACT-Lite column, the questions are:

* Does the 7B judge still flag nearly every fact, or does the union stop collapsing onto the permissive judge? (Judge ablation section: `precision_vs_ensemble` for each judge.)
* Does the incomplete rate fall further than 0.091?
* Does the gap between union and majority close? On the laptop, majority was far better (0.042 vs 0.091); if better judges close that gap, the paper's union rule is vindicated and the laptop finding was an artifact of weak judges.

## 7 · Experiment B — full upgrade

Everything moves to 7–8B, including extraction and verification, so the baseline changes too. Results land in `outputs_colab_b/`. Longer: nothing replays from cache except retrieval.

In [ ]:
CFG_B = "colab/config.full7b.yaml"

t0 = time.time()
!python -u -m experiments.run_ablation --config {CFG_B} {LIMIT}
!python -u -m experiments.run_ablation --config {CFG_B} --ensemble majority {LIMIT}
!python -u -m experiments.run_baseline --config {CFG_B} {LIMIT}
!python -u -m experiments.run_verifact --config {CFG_B} {LIMIT}
!python -u -m experiments.compare --config {CFG_B}
print(f"\nexperiment B took {(time.time()-t0)/60:.1f} min")

In [ ]:
!python colab/compare_runs.py laptop-3B=outputs/summary.json judges-7B=outputs_colab_a/summary.json full-7B=outputs_colab_b/summary.json

## 8 · Download the results

Takes back the output directories and the caches, so the Colab calls do not have to be paid for twice and the laptop can regenerate every table offline.

In [ ]:
import zipfile, pathlib
from google.colab import files

OUT = pathlib.Path("/content/verifact_lite_colab_results.zip")
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for folder in ["outputs_colab_a", "outputs_colab_b", "cache", "data/evidence"]:
        base = pathlib.Path(folder)
        if not base.exists():
            continue
        for path in base.rglob("*"):
            if path.is_file():
                zf.write(path, path.as_posix())

print(f"{OUT.stat().st_size / 1048576:.1f} MB")
files.download(str(OUT))

Unzip it over the repo on the laptop. `outputs_colab_a/` and `outputs_colab_b/` then sit beside `outputs/`, and

```bash
python colab/compare_runs.py laptop-3B=outputs/summary.json judges-7B=outputs_colab_a/summary.json full-7B=outputs_colab_b/summary.json
```

reproduces the side-by-side table without a GPU.

**A caution for the write-up.** Experiment A is a controlled comparison and belongs beside the current results. Experiment B is a different system — its baseline is not the laptop baseline, so its precision and F1 cannot be compared with the laptop's numbers directly, only with its own baseline column.